# 1. Reranking Baseline: Causal Popularity

A transparent, non-personalised reference point. Every candidate in a logged impression
is ranked by its **smoothed click-through rate computed only from events before the
impression**. There are no trainable parameters, so the same rule is evaluated on
validation and then on test.

This notebook also fixes the contract every learned reranker follows: write one score
per logged candidate, then evaluate complete logged slates with `rr_evaluation.py`.

## 1.1 Load Packages

In [ ]:
import os
import sys
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq
from IPython.display import display

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()
else:
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    while (
        not (PROJECT_ROOT / "5_Models").exists() and PROJECT_ROOT != PROJECT_ROOT.parent
    ):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = min((RUN / f"ranking_features_{split}").glob("*.parquet"))
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Model: Smoothed Prior Article CTR

For candidate article $a$ at impression time $t$:

$$s(a,t) = \frac{c_{a,<t} + \alpha\, p_{<t}}{e_{a,<t} + \alpha}$$

- $c_{a,<t}$ and $e_{a,<t}$ are the article's earlier clicks and exposures
- $p_{<t}$ is the expanding global CTR prior
- $\alpha$ is the smoothing strength set by the feature run

Feature engineering already stores this value as `smoothed_prior_article_ctr`, so the
model simply reads it; nothing here can see events after the impression.

```text
Earlier article clicks and exposures + earlier global CTR
                         ↓
                Smoothed article CTR  →  candidate score
```

In [ ]:
MODEL_NAME = "popularity_smoothed_prior_article_ctr"
OUTPUT_DIR = ARTIFACTS / "popularity"


def popularity_score(frame):
    """Score each candidate by its causal smoothed article CTR."""
    return frame["smoothed_prior_article_ctr"].to_numpy()

# 3. Evaluation

The rule has no settings to tune, so validation and test are both reported. The
evaluator rejects incomplete or duplicate score files and breaks tied scores by
ascending `news_code`.

In [ ]:
popularity_results = []
for split in ("validation", "test"):
    result, _ = score_and_evaluate(
        popularity_score,
        split=split,
        feature_run=RUN,
        output_dir=OUTPUT_DIR,
        model_name=MODEL_NAME,
        columns=["smoothed_prior_article_ctr"],
        results_dir=RESULTS,
    )
    popularity_results.append(result)

popularity_results = pl.concat(popularity_results)
display(
    popularity_results.select(
        "model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"
    )
)

# 4. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(
    example_user,
    score_file=OUTPUT_DIR / "validation_scores.parquet",
    feature_run=RUN,
    top_n=5,
)